# Halo Occupation Distribution as a function of $\tau_0$

The HOD ⟨$N_\mathrm{gal}\,|\,M_h$⟩ is the most direct test of where dynamical friction matters in the high-$M_h$ regime. Two limits:

* $\tau_0 = 0$ (instant merging): every infalling satellite is destroyed, so $\langle N_\mathrm{sat}\,|\,M_h\rangle$ is suppressed. $\langle N_\mathrm{cen}\rangle$ stays at 1 by construction in occupied halos.
* $\tau_0 \to \infty$ (no merging): every galaxy that ever entered a halo is still there, so $\langle N_\mathrm{sat}\,|\,M_h\rangle$ is enhanced — particularly at large $M_h$ where merger histories are rich.

The signature in $\langle N_\mathrm{sat}\rangle(M_h)$ should grow with $M_h$ because the satellite population scales (very roughly) as $M_h$. So the *fractional* effect of $\tau_0$ on $\langle N_\mathrm{sat}\rangle$ is the cleanest probe — and it should remain large at $\log_{10}M_h \sim 14$.

## Method

* **Two-histogram HOD.** Numerator: galaxy counts (above a stellar-mass cut) binned by their host-halo mass $M_\mathrm{hhalo}$. Denominator: FOF group counts from `Trees/mphalo` (so empty halos contribute zero correctly).
* **FOF-central definition.** A galaxy is the central of its FOF group if `is_central == 1` and `mhalo / mhhalo > 0.5`. Satellites = total − FOF centrals.
* **Stellar-mass selection.** Default is $M_\star \geq 10^9\,M_\odot/h$; the second figure shows how the result depends on the cut.

All masses in $M_\odot/h$.

In [ ]:
import sys
from pathlib import Path

_src = str(Path('../../src').resolve())
_here = str(Path.cwd())
for _p in [_src, _here]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

import matplotlib.pyplot as plt
import numpy as np
from utils.matplotlib_config import setconfig
from tau0_helpers import (
    DEFAULT_IVOLS,
    RUNS,
    RUN_LABELS,
    SNAPSHOTS,
    collect_per_ivol,
    hod_per_ivol,
    safe_ratio,
    stack_per_ivol,
    style_for,
)

setconfig({
    'figure': {'figsize': (8, 6)},
    'font': {'size': 14},
    'legend': {'fontsize': 12, 'title_fontsize': 13},
    'axes': {'labelsize': 14},
})

halo_bins = np.arange(11.0, 15.51, 0.20)
halo_centers = 0.5 * (halo_bins[1:] + halo_bins[:-1])
ivols = DEFAULT_IVOLS

DEFAULT_MSTAR_CUT = 1e9
from tau0_helpers import (
    central_shmr_per_ivol,
    fof_central_mask,
    load_galaxy_fields,
    smf_split_per_ivol,
)


## Compute per-ivol HOD for every (run, snapshot) at the default $M_\star$ cut

In [ ]:
def make_summariser(mstar_min):
    return lambda d: hod_per_ivol(d, halo_bins, mstar_min=mstar_min)

stacked = {}
for snapshot in SNAPSHOTS:
    stacked[snapshot] = {}
    for run_label, run_path in RUNS.items():
        summaries = collect_per_ivol(run_path, snapshot, ivols, make_summariser(DEFAULT_MSTAR_CUT))
        stacked[snapshot][run_label] = stack_per_ivol(
            summaries,
            keys=('n_total', 'n_cen', 'n_sat', 'halo_counts'),
            nboot=500,
            seed=37,
        )
        n = stacked[snapshot][run_label]['n_used']
        print(f"{snapshot} {run_label:>10s}: {n:2d} ivols used")

## Figure 1 — Mean occupation $\langle N\,|\,M_h\rangle$

Three columns: total / centrals / satellites. Two rows for the two snapshots. Bands are bootstrap 16-84. Bins below the dashed grey line have fewer than 5 halos in the per-ivol sample (suppress those because the noise dominates).

In [ ]:
components = [
    ('n_total', r'$\langle N_\mathrm{total}\rangle$'),
    ('n_cen', r'$\langle N_\mathrm{cen}\rangle$'),
    ('n_sat', r'$\langle N_\mathrm{sat}\rangle$'),
]

min_halos = 3.0  # per-ivol mean halo count required to plot a bin

fig, axes = plt.subplots(
    len(SNAPSHOTS), len(components),
    figsize=(13, 4.6 * len(SNAPSHOTS)),
    sharex=True, sharey=True,
)
if len(SNAPSHOTS) == 1:
    axes = axes[None, :]

for i, snapshot in enumerate(SNAPSHOTS):
    z_val = SNAPSHOTS[snapshot][1]
    for j, (key, comp_label) in enumerate(components):
        ax = axes[i, j]
        for run_label in RUNS:
            s = stacked[snapshot][run_label]
            if s['n_used'] == 0 or key not in s:
                continue
            n_mean = s[key]['mean']
            lo = s[key]['boot_lo']
            hi = s[key]['boot_hi']
            n_halos = s['halo_counts']['mean']
            ok = np.isfinite(n_mean) & (n_halos >= min_halos)
            st = style_for(run_label)
            ax.plot(halo_centers[ok], np.maximum(n_mean[ok], 1e-3), '-', lw=2.2, **st)
            band_ok = ok & (lo > 0) & (hi > 0)
            ax.fill_between(
                halo_centers[band_ok], lo[band_ok], hi[band_ok],
                color=st['color'], alpha=0.20, linewidth=0,
            )
        ax.set_yscale('log')
        ax.set_ylim(5e-3, 200.0)
        ax.set_xlim(11.2, 15.0)
        ax.grid(True, alpha=0.3, which='both')
        if i == 0:
            ax.set_title(comp_label)
        if i == len(SNAPSHOTS) - 1:
            ax.set_xlabel(r'$\log_{10}\,M_\mathrm{halo}\ [M_\odot/h]$')
        if j == 0:
            ax.set_ylabel(r'$\langle N\,|\,M_h\rangle$' + f'\nz = {z_val:.1f}')
        if i == 0 and j == 0:
            ax.legend(loc='upper left', fontsize=10)
fig.suptitle(r'HOD across $\tau_0$ variants  ($M_\star \geq 10^9\,M_\odot/h$)', y=1.0)
plt.tight_layout()
plt.show()

## Figure 2 — Ratios relative to Default

The fractional shift in $\langle N_\mathrm{sat}\rangle$ is the headline diagnostic for $\tau_0$. We expect:

* $\tau_0=0$: ratio $\lesssim 1$, growing weaker (closer to 1) at large $M_h$ as the merger time becomes negligible regardless of $\tau_0$.
* $\tau_0=\infty$: ratio $> 1$, growing with $M_h$ as more never-merged satellites accumulate.

In [ ]:
fig, axes = plt.subplots(
    len(SNAPSHOTS), len(components),
    figsize=(13, 4.0 * len(SNAPSHOTS)),
    sharex=True, sharey=True,
)
if len(SNAPSHOTS) == 1:
    axes = axes[None, :]

for i, snapshot in enumerate(SNAPSHOTS):
    z_val = SNAPSHOTS[snapshot][1]
    for j, (key, comp_label) in enumerate(components):
        ax = axes[i, j]
        s_def = stacked[snapshot]['Default']
        if s_def['n_used'] == 0:
            continue
        n_def = s_def[key]['mean']
        n_halos = s_def['halo_counts']['mean']
        for run_label in RUNS:
            if run_label == 'Default':
                continue
            s = stacked[snapshot][run_label]
            ratio = safe_ratio(s[key]['mean'], n_def)
            ratio_lo = safe_ratio(s[key]['boot_lo'], n_def)
            ratio_hi = safe_ratio(s[key]['boot_hi'], n_def)
            ok = np.isfinite(ratio) & (n_halos >= min_halos) & (n_def > 1e-3)
            st = style_for(run_label)
            ax.plot(halo_centers[ok], ratio[ok], '-', lw=2.2, **st)
            ax.fill_between(
                halo_centers[ok], ratio_lo[ok], ratio_hi[ok],
                color=st['color'], alpha=0.20, linewidth=0,
            )
        ax.axhline(1.0, color='0.4', ls=':', lw=1)
        ax.set_yscale('log')
        ax.set_ylim(0.1, 30.0)
        ax.set_yticks([0.1, 0.3, 1.0, 3.0, 10.0])
        ax.set_yticklabels(['0.1', '0.3', '1', '3', '10'])
        ax.set_xlim(11.5, 15.0)
        ax.grid(True, alpha=0.3, which='both')
        if i == 0:
            ax.set_title(comp_label)
        if i == len(SNAPSHOTS) - 1:
            ax.set_xlabel(r'$\log_{10}\,M_\mathrm{halo}\ [M_\odot/h]$')
        if j == 0:
            ax.set_ylabel(r'$N/N^\mathrm{default}$' + f'\nz = {z_val:.1f}')
        if i == 0 and j == 0:
            ax.legend(loc='upper left', fontsize=10)
fig.suptitle(r'HOD ratios — $\tau_0$ moves satellite occupancy at fixed $M_h$', y=1.0)
plt.tight_layout()
plt.show()

## Figure 3 — Power-law fit to $\langle N_\mathrm{sat}\rangle(M_h)$ at large $M_h$

In the cluster regime $\langle N_\mathrm{sat}\rangle \propto M_h^\alpha$ with $\alpha \sim 1$. $\tau_0$ should change the *normalisation* and possibly the slope $\alpha$. We fit $\log_{10}\langle N_\mathrm{sat}\rangle = \alpha\,\log_{10}M_h + \beta$ on bins between $\log_{10}M_h = 12.5$ and the highest reliably-populated bin.

In [ ]:
import pandas as pd

fit_lo, fit_hi = 12.5, 15.0
min_n_for_fit = 0.5

fit_rows = []
fig, axes = plt.subplots(1, len(SNAPSHOTS), figsize=(11, 5.0), sharey=True)
if len(SNAPSHOTS) == 1:
    axes = [axes]

for j, snapshot in enumerate(SNAPSHOTS):
    z_val = SNAPSHOTS[snapshot][1]
    ax = axes[j]
    for run_label in RUNS:
        s = stacked[snapshot][run_label]
        if s['n_used'] == 0:
            continue
        n_sat = s['n_sat']['mean']
        n_halos = s['halo_counts']['mean']
        st = style_for(run_label)
        ok = np.isfinite(n_sat) & (n_sat > min_n_for_fit) & (halo_centers >= fit_lo) & (halo_centers <= fit_hi) & (n_halos >= min_halos)
        ax.plot(halo_centers[ok], n_sat[ok], 'o', ms=6, mec=st['color'], mfc='none', mew=1.5)
        if ok.sum() >= 3:
            x = halo_centers[ok]
            y = np.log10(n_sat[ok])
            alpha, beta = np.polyfit(x, y, 1)
            x_fine = np.linspace(x.min(), x.max(), 50)
            ax.plot(x_fine, 10 ** (alpha * x_fine + beta), '-', lw=2.2,
                    color=st['color'],
                    label=f"{st['label']}: $\\alpha = {alpha:.2f}$")
            fit_rows.append({
                'snapshot': snapshot, 'z': z_val, 'run': run_label,
                'alpha': round(alpha, 3), 'log10_normalization_at_logMh_13': round(alpha * 13.0 + beta, 3),
                'n_bins_used': int(ok.sum()),
            })
    ax.set_yscale('log')
    ax.set_xlim(12.3, 15.0)
    ax.set_ylim(0.3, 200.0)
    ax.set_xlabel(r'$\log_{10}\,M_\mathrm{halo}\ [M_\odot/h]$')
    ax.set_title(f'{snapshot}  (z = {z_val:.1f})')
    ax.grid(True, alpha=0.3, which='both')
    if j == 0:
        ax.set_ylabel(r'$\langle N_\mathrm{sat}\,|\,M_h\rangle$')
    ax.legend(loc='upper left', fontsize=10)
fig.suptitle(r'Satellite HOD in the high-mass regime: power-law fits', y=1.02)
plt.tight_layout()
plt.show()

fit_df = pd.DataFrame(fit_rows)
fit_df

## Figure 4 — Stellar-mass-cut sensitivity

Whether the τ₀ effect grows or shrinks with the threshold tells you which satellite populations dynamical friction operates on. We sweep the cut from $10^9$ to $10^{11}\,M_\odot/h$ at $z=0$.

In [ ]:
snapshot = 'iz271'
z_val = SNAPSHOTS[snapshot][1]
mstar_cuts = [1e9, 1e10, 1e11]

fig, axes = plt.subplots(1, len(mstar_cuts), figsize=(13, 4.6), sharex=True, sharey=True)
for j, mstar_cut in enumerate(mstar_cuts):
    ax = axes[j]
    s_def = None
    cuts_results = {}
    for run_label, run_path in RUNS.items():
        summaries = collect_per_ivol(
            run_path, snapshot, ivols, make_summariser(mstar_cut)
        )
        s = stack_per_ivol(
            summaries,
            keys=('n_sat', 'halo_counts'),
            nboot=300,
            seed=41,
        )
        cuts_results[run_label] = s
    s_def = cuts_results['Default']
    n_def = s_def['n_sat']['mean']
    n_halos = s_def['halo_counts']['mean']
    for run_label in RUNS:
        if run_label == 'Default':
            continue
        s = cuts_results[run_label]
        ratio = safe_ratio(s['n_sat']['mean'], n_def)
        ok = np.isfinite(ratio) & (n_halos >= min_halos) & (n_def > 1e-3)
        st = style_for(run_label)
        ax.plot(halo_centers[ok], ratio[ok], '-', lw=2.2, **st)
    ax.axhline(1.0, color='0.4', ls=':', lw=1)
    ax.set_yscale('log')
    ax.set_ylim(0.1, 30.0)
    ax.set_xlim(11.5, 15.0)
    ax.set_yticks([0.1, 0.3, 1.0, 3.0, 10.0])
    ax.set_yticklabels(['0.1', '0.3', '1', '3', '10'])
    ax.grid(True, alpha=0.3, which='both')
    ax.set_xlabel(r'$\log_{10}\,M_\mathrm{halo}\ [M_\odot/h]$')
    ax.set_title(rf'$M_\star \geq 10^{{{int(np.log10(mstar_cut))}}}$ $M_\odot/h$')
    if j == 0:
        ax.set_ylabel(r'$\langle N_\mathrm{sat}\rangle/\langle N_\mathrm{sat}\rangle^\mathrm{default}$')
        ax.legend(loc='upper left', fontsize=10)
fig.suptitle(
    f'How the $\\tau_0$ effect on satellite HOD depends on the stellar-mass cut  ($z={z_val:.1f}$)',
    y=1.02,
)
plt.tight_layout()
plt.show()

## Quantitative summary

Halo masses where the τ₀ effect on $\langle N_\mathrm{sat}\rangle$ peaks, and the size of the bracket at three diagnostic masses.

In [ ]:
rows = []
for snapshot in SNAPSHOTS:
    z_val = SNAPSHOTS[snapshot][1]
    s_def = stacked[snapshot]['Default']
    if s_def['n_used'] == 0:
        continue
    for log_mh_target in (12.0, 13.0, 14.0):
        i = int(np.argmin(np.abs(halo_centers - log_mh_target)))
        n_def_sat = s_def['n_sat']['mean'][i]
        n_halos = s_def['halo_counts']['mean'][i]
        for run_label in RUNS:
            s = stacked[snapshot][run_label]
            if s['n_used'] == 0:
                continue
            row = {
                'snapshot': snapshot,
                'z': z_val,
                'log10_Mhalo': log_mh_target,
                'run': run_label,
                'N_total': round(s['n_total']['mean'][i], 3),
                'N_cen': round(s['n_cen']['mean'][i], 3),
                'N_sat': round(s['n_sat']['mean'][i], 3),
                'N_sat/default': round(safe_ratio(np.array([s['n_sat']['mean'][i]]), np.array([n_def_sat]))[0], 3) if n_def_sat > 0 else np.nan,
                'n_halos_per_ivol': int(n_halos),
            }
            rows.append(row)
df = pd.DataFrame(rows)
df

## Interpretation

* **$\langle N_\mathrm{cen}\rangle$ saturates at 1** in halos massive enough to host a galaxy above the threshold, irrespective of $\tau_0$. The small gap between runs at intermediate $M_h$ is a *side-effect* of the central-mass shift: when $\tau_0=\infty$, the central is starved and may fail the $M_\star$ cut.
* **$\langle N_\mathrm{sat}\rangle$ is the headline diagnostic.** The $\tau_0=\infty$ run inflates the satellite occupancy at every halo mass (Fig 2 right column); the asymmetry between Default and $\tau_0=0$ tells you how close the Default already is to the instant-merging limit.
* **Power-law slope $\alpha$ (Fig 3)** can shift between runs if dynamical friction is more important for low-mass satellites (which take longer to spiral in than they would in the $\tau_0=0$ limit).
* **$M_\star$-cut sensitivity (Fig 4)** distinguishes whether the $\tau_0$ effect is dominated by low-mass or high-mass satellites. If the gap shrinks with rising cut, the $\tau_0$ knob is acting mostly on small satellites; if it grows, it's the massive ones that fail to merge.

---
## 2. Central Stellar-to-Halo Mass Relation

The most direct stellar-mass-budget consequence of $\tau_0$ is on the **brightest central galaxy (BCG)** — the central of the dominant subhalo of each FOF group. Two limits:

* $\tau_0 \to 0$: every infalling satellite merges with the BCG immediately, boosting BCG mass.
* $\tau_0 \to \infty$: satellites never merge; the BCG is starved of accreted stellar mass.

If dynamical friction matters at high $M_\mathrm{halo}$, the bracket between these limits is visible in the central SHMR above $\log_{10}(M_\mathrm{halo}/[M_\odot/h]) \sim 13$. The **FOF-central** definition: `is_central==1` AND `mhalo/mhhalo > 0.5`.

In [ ]:
halo_bins_shmr = np.arange(11.0, 15.51, 0.25)
halo_centers_shmr = 0.5 * (halo_bins_shmr[1:] + halo_bins_shmr[:-1])
summarise_shmr = lambda d: central_shmr_per_ivol(d, halo_bins_shmr)

stacked_shmr = {}
for snapshot in SNAPSHOTS:
    stacked_shmr[snapshot] = {}
    for run_label, run_path in RUNS.items():
        summaries = collect_per_ivol(run_path, snapshot, ivols, summarise_shmr)
        stacked_shmr[snapshot][run_label] = stack_per_ivol(
            summaries,
            keys=('median', 'p16', 'p84', 'counts'),
            nboot=500,
            seed=23,
        )
        n = stacked_shmr[snapshot][run_label]['n_used']
        print(f'{snapshot} {run_label:>10s}: {n:2d} ivols used')

### Figure 5 — Central SHMR with bootstrap range and ratio panel

Two columns (z=0, z=0.5). Top: median $\log_{10}(M_\star)$ vs $\log_{10}(M_\mathrm{halo})$ with bootstrap 16–84 band. Dotted lines show the Default 16/84 *intrinsic* scatter. Bottom: $M_\star^\mathrm{model}/M_\star^\mathrm{default}$ as a linear ratio.

In [ ]:
min_count_shmr = 30

fig, axes = plt.subplots(
    2, len(SNAPSHOTS),
    figsize=(11, 8),
    sharex='col',
    gridspec_kw={'height_ratios': [3, 1.5], 'hspace': 0.05, 'wspace': 0.05},
)
if len(SNAPSHOTS) == 1:
    axes = axes[:, None]

for j, snapshot in enumerate(SNAPSHOTS):
    z_val = SNAPSHOTS[snapshot][1]
    ax_main = axes[0, j]
    ax_ratio = axes[1, j]
    default_med = stacked_shmr[snapshot]['Default']['median']['mean']

    for run_label in RUNS:
        s = stacked_shmr[snapshot][run_label]
        if s['n_used'] == 0:
            continue
        median_mean = s['median']['mean']
        boot_lo = s['median']['boot_lo']
        boot_hi = s['median']['boot_hi']
        n_bin_mean = s['counts']['mean']
        ok = np.isfinite(median_mean) & (n_bin_mean >= min_count_shmr)
        st = style_for(run_label)
        ax_main.plot(halo_centers_shmr[ok], median_mean[ok], '-', lw=2.2, **st)
        ax_main.fill_between(
            halo_centers_shmr[ok], boot_lo[ok], boot_hi[ok],
            color=st['color'], alpha=0.20, linewidth=0,
        )
        if run_label == 'Default':
            p16_m = s['p16']['mean']
            p84_m = s['p84']['mean']
            ax_main.plot(halo_centers_shmr[ok], p16_m[ok], ':', color=st['color'], lw=1, alpha=0.7)
            ax_main.plot(halo_centers_shmr[ok], p84_m[ok], ':', color=st['color'], lw=1, alpha=0.7,
                         label='Default 16/84 scatter')
        if run_label == 'Default':
            continue
        ratio = 10 ** (median_mean - default_med)
        ratio_lo = 10 ** (boot_lo - default_med)
        ratio_hi = 10 ** (boot_hi - default_med)
        ax_ratio.plot(halo_centers_shmr[ok], ratio[ok], '-', lw=2.2, **st)
        ax_ratio.fill_between(
            halo_centers_shmr[ok], ratio_lo[ok], ratio_hi[ok],
            color=st['color'], alpha=0.20, linewidth=0,
        )

    ax_main.set_title(f'{snapshot}  (z = {z_val:.1f})')
    ax_main.set_ylabel(r'$\log_{10}\,M_{\star,\,\mathrm{cen}}\ [M_\odot/h]$')
    ax_main.set_ylim(8.5, 12.5)
    ax_main.grid(True, alpha=0.3)
    ax_main.legend(loc='upper left', fontsize=10, frameon=False)

    ax_ratio.axhline(1.0, color='0.4', ls=':', lw=1)
    ax_ratio.set_xlabel(r'$\log_{10}\,M_\mathrm{halo}\ [M_\odot/h]$')
    ax_ratio.set_xlim(11.0, 15.3)
    ax_ratio.set_ylim(0.1, 10.0)
    ax_ratio.set_yscale('log')
    ax_ratio.set_yticks([0.1, 0.3, 1.0, 3.0])
    ax_ratio.set_yticklabels(['0.1', '0.3', '1', '3'])
    ax_ratio.grid(True, alpha=0.3, which='both')
    if j == 0:
        ax_ratio.set_ylabel(r'$M_\star\,/\,M_\star^\mathrm{default}$')

fig.suptitle(r'FOF-central SHMR across $\tau_0$ variants', y=0.995)
plt.tight_layout()
plt.show()

### Figure 6 — BCG stellar-mass distribution at fixed $M_\mathrm{halo}$ (z=0)

Pooling all 16 ivols per run at two mass scales shows whether $\tau_0$ shifts the *median* or reshapes the *tail* of the BCG distribution.

In [ ]:
snapshot_bcg = 'iz271'
z_val_bcg = SNAPSHOTS[snapshot_bcg][1]
halo_bin_edges = [
    (11.7, 12.3, r'MW scale  $\log_{10}M_h \in [11.7, 12.3]$'),
    (13.7, 14.3, r'Group scale  $\log_{10}M_h \in [13.7, 14.3]$'),
]
ms_bins_bcg = np.arange(8.5, 12.51, 0.1)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharey=True)
for ax, (lo, hi, title) in zip(axes, halo_bin_edges):
    for run_label, run_path in RUNS.items():
        log_ms_pool = []
        for iv in ivols:
            data = load_galaxy_fields(run_path / snapshot_bcg, iv)
            if data is None:
                continue
            mask = fof_central_mask(data) & (data['mstar'] > 0)
            log_mh = np.log10(data['mhhalo'][mask])
            log_ms = np.log10(data['mstar'][mask])
            in_bin = (log_mh >= lo) & (log_mh < hi)
            log_ms_pool.append(log_ms[in_bin])
        if not log_ms_pool:
            continue
        log_ms_all = np.concatenate(log_ms_pool)
        st = style_for(run_label)
        ax.hist(log_ms_all, bins=ms_bins_bcg, density=True,
                histtype='step', lw=2.2, color=st['color'],
                label=f"{st['label']}  (N={log_ms_all.size:,})")
        ax.axvline(np.median(log_ms_all), color=st['color'], lw=1.0, ls='--', alpha=0.6)
    ax.set_title(title)
    ax.set_xlabel(r'$\log_{10}\,M_{\star,\,\mathrm{cen}}\ [M_\odot/h]$')
    ax.legend(fontsize=9, loc='upper left')
    ax.grid(True, alpha=0.3)
axes[0].set_ylabel('PDF')
fig.suptitle(f'BCG stellar-mass distribution at fixed $M_\\mathrm{{halo}}$  (z = {z_val_bcg:.1f})',
             y=1.02)
plt.tight_layout()
plt.show()

### Figure 7 — Logarithmic scatter $\sigma(\log M_\star\,|\,M_h)$

$\tau_0$ should not only shift the mean SHMR but also broaden/narrow the per-halo scatter, because merger histories vary from halo to halo. We use $(p_{84}-p_{16})/2$ in dex (robust 1-σ estimator), averaged across ivols.

In [ ]:
fig, axes = plt.subplots(1, len(SNAPSHOTS), figsize=(11, 4.5), sharey=True, sharex=True)
if len(SNAPSHOTS) == 1:
    axes = [axes]

for j, snapshot in enumerate(SNAPSHOTS):
    z_val = SNAPSHOTS[snapshot][1]
    ax = axes[j]
    for run_label in RUNS:
        s = stacked_shmr[snapshot][run_label]
        if s['n_used'] == 0:
            continue
        scatter = 0.5 * (s['p84']['mean'] - s['p16']['mean'])
        n_bin_mean = s['counts']['mean']
        ok = np.isfinite(scatter) & (n_bin_mean >= min_count_shmr)
        st = style_for(run_label)
        ax.plot(halo_centers_shmr[ok], scatter[ok], '-', lw=2.2, **st)
    ax.set_title(f'{snapshot}  (z = {z_val:.1f})')
    ax.set_xlabel(r'$\log_{10}\,M_\mathrm{halo}\ [M_\odot/h]$')
    ax.set_xlim(11.0, 15.3)
    ax.set_ylim(0.0, 1.0)
    ax.grid(True, alpha=0.3)
    if j == 0:
        ax.set_ylabel(r'$\sigma(\log_{10} M_\star\,|\,M_h)$  [dex]')
    ax.legend(fontsize=10)
fig.suptitle(r'BCG scatter $\sigma(\log M_\star\,|\,M_h)$ across $\tau_0$ variants', y=1.02)
plt.tight_layout()
plt.show()

---
## 3. Stellar Mass Function

$\tau_0$ is a **redistribution knob**: it does not change the total stellar mass produced (set by gas physics and feedback) but only *where* that mass ends up — in centrals (BCGs) or in satellites. This makes the SMF a uniquely diagnostic test:

* The **total SMF** should be approximately invariant under $\tau_0$ changes.
* The **central SMF** should bracket between $\tau_0=0$ (BCG accretes all) and $\tau_0\to\infty$ (BCG is starved) at $\log_{10}M_\star \gtrsim 11$.
* The **satellite SMF** should move oppositely.

All masses in $M_\odot/h$; $\Phi$ in $(\mathrm{Mpc}/h)^{-3}\,\mathrm{dex}^{-1}$.

In [ ]:
mstar_bins = np.arange(8.0, 12.61, 0.15)
mstar_centers = 0.5 * (mstar_bins[1:] + mstar_bins[:-1])
summarise_smf = lambda d: smf_split_per_ivol(d, mstar_bins)

stacked_smf = {}
for snapshot in SNAPSHOTS:
    stacked_smf[snapshot] = {}
    for run_label, run_path in RUNS.items():
        summaries = collect_per_ivol(run_path, snapshot, ivols, summarise_smf)
        stacked_smf[snapshot][run_label] = stack_per_ivol(
            summaries,
            keys=('phi_total', 'phi_cen', 'phi_sat'),
            nboot=500,
            seed=29,
        )
        n = stacked_smf[snapshot][run_label]['n_used']
        print(f'{snapshot} {run_label:>10s}: {n:2d} ivols used')

### Figure 8 — Total / central / satellite SMF

Three columns, two snapshot rows. The leftmost (total) column should be nearly invariant across runs — confirming that $\tau_0$ is a redistribution knob.

In [ ]:
smf_components = [
    ('phi_total', 'Total'),
    ('phi_cen',   'Centrals'),
    ('phi_sat',   'Satellites'),
]

fig, axes = plt.subplots(
    len(SNAPSHOTS), len(smf_components),
    figsize=(13, 4.6 * len(SNAPSHOTS)),
    sharex=True, sharey=True,
)
if len(SNAPSHOTS) == 1:
    axes = axes[None, :]

for i, snapshot in enumerate(SNAPSHOTS):
    z_val = SNAPSHOTS[snapshot][1]
    for j, (key, comp_label) in enumerate(smf_components):
        ax = axes[i, j]
        for run_label in RUNS:
            s = stacked_smf[snapshot][run_label]
            if s['n_used'] == 0 or key not in s:
                continue
            phi = s[key]['mean']
            lo = s[key]['boot_lo']
            hi = s[key]['boot_hi']
            ok = np.isfinite(phi) & (phi > 0)
            st = style_for(run_label)
            ax.plot(mstar_centers[ok], phi[ok], '-', lw=2.2, **st)
            band_ok = ok & (lo > 0) & (hi > 0)
            ax.fill_between(
                mstar_centers[band_ok], lo[band_ok], hi[band_ok],
                color=st['color'], alpha=0.20, linewidth=0,
            )
        ax.set_yscale('log')
        ax.set_ylim(1e-6, 0.2)
        ax.set_xlim(8.5, 12.4)
        ax.grid(True, alpha=0.3, which='both')
        if i == 0:
            ax.set_title(comp_label)
        if i == len(SNAPSHOTS) - 1:
            ax.set_xlabel(r'$\log_{10}\,M_\star\ [M_\odot/h]$')
        if j == 0:
            ax.set_ylabel(r'$\Phi\ [(\mathrm{Mpc}/h)^{-3}\,\mathrm{dex}^{-1}]$'
                          + f'\nz = {z_val:.1f}')
        if i == 0 and j == 0:
            ax.legend(loc='lower left', fontsize=10)
fig.suptitle(r'Stellar mass function across $\tau_0$ variants', y=1.0)
plt.tight_layout()
plt.show()

### Figure 9 — SMF ratios relative to Default

Log-axis ratios isolate the $\tau_0$ effect. The total panel should hover near unity; the central panel shows the BCG-mass redistribution signal.

In [ ]:
fig, axes = plt.subplots(
    len(SNAPSHOTS), len(smf_components),
    figsize=(13, 4.0 * len(SNAPSHOTS)),
    sharex=True, sharey=True,
)
if len(SNAPSHOTS) == 1:
    axes = axes[None, :]

for i, snapshot in enumerate(SNAPSHOTS):
    z_val = SNAPSHOTS[snapshot][1]
    for j, (key, comp_label) in enumerate(smf_components):
        ax = axes[i, j]
        s_def = stacked_smf[snapshot]['Default']
        if s_def['n_used'] == 0 or key not in s_def:
            continue
        phi_def = s_def[key]['mean']
        for run_label in RUNS:
            if run_label == 'Default':
                continue
            s = stacked_smf[snapshot][run_label]
            if s['n_used'] == 0 or key not in s:
                continue
            ratio = safe_ratio(s[key]['mean'], phi_def)
            ratio_lo = safe_ratio(s[key]['boot_lo'], phi_def)
            ratio_hi = safe_ratio(s[key]['boot_hi'], phi_def)
            ok = np.isfinite(ratio) & (mstar_centers >= 9.5) & (phi_def > 1e-7)
            st = style_for(run_label)
            ax.plot(mstar_centers[ok], ratio[ok], '-', lw=2.2, **st)
            ax.fill_between(
                mstar_centers[ok], ratio_lo[ok], ratio_hi[ok],
                color=st['color'], alpha=0.20, linewidth=0,
            )
        ax.axhline(1.0, color='0.4', ls=':', lw=1)
        ax.set_yscale('log')
        ax.set_ylim(0.05, 20.0)
        ax.set_yticks([0.1, 0.3, 1.0, 3.0, 10.0])
        ax.set_yticklabels(['0.1', '0.3', '1', '3', '10'])
        ax.set_xlim(9.5, 12.3)
        ax.grid(True, alpha=0.3, which='both')
        if i == 0:
            ax.set_title(comp_label)
        if i == len(SNAPSHOTS) - 1:
            ax.set_xlabel(r'$\log_{10}\,M_\star\ [M_\odot/h]$')
        if j == 0:
            ax.set_ylabel(r'$\Phi/\Phi^\mathrm{default}$' + f'\nz = {z_val:.1f}')
        if i == 0 and j == 0:
            ax.legend(loc='upper left', fontsize=10)
fig.suptitle(r'SMF ratios relative to default $\tau_0$', y=1.0)
plt.tight_layout()
plt.show()

### Figure 10 — Satellite fraction and cumulative central abundance

$f_\mathrm{sat} = \Phi_\mathrm{sat}/\Phi_\mathrm{total}$ at fixed $M_\star$ is independent of normalisation — the cleanest direct readout of how $\tau_0$ partitions the population. The cumulative panel ($N(>M_\star)$ for centrals) is directly comparable to BCG counts in cluster surveys.

In [ ]:
dlog_ms = np.diff(mstar_bins)

def cumulative_above(phi):
    return np.cumsum((phi * dlog_ms)[::-1])[::-1]

fig, axes = plt.subplots(len(SNAPSHOTS), 2, figsize=(11, 4.6 * len(SNAPSHOTS)),
                         sharex='col', sharey='col')
if len(SNAPSHOTS) == 1:
    axes = axes[None, :]

for i, snapshot in enumerate(SNAPSHOTS):
    z_val = SNAPSHOTS[snapshot][1]
    ax_fsat = axes[i, 0]
    ax_cum  = axes[i, 1]
    for run_label in RUNS:
        s = stacked_smf[snapshot][run_label]
        if s['n_used'] == 0:
            continue
        st = style_for(run_label)
        f_sat = safe_ratio(s['phi_sat']['mean'], s['phi_total']['mean'])
        f_sat_lo = safe_ratio(s['phi_sat']['boot_lo'], s['phi_total']['mean'])
        f_sat_hi = safe_ratio(s['phi_sat']['boot_hi'], s['phi_total']['mean'])
        ok = np.isfinite(f_sat) & (s['phi_total']['mean'] > 1e-7) & (mstar_centers >= 9.0)
        ax_fsat.plot(mstar_centers[ok], f_sat[ok], '-', lw=2.2, **st)
        ax_fsat.fill_between(mstar_centers[ok], f_sat_lo[ok], f_sat_hi[ok],
                             color=st['color'], alpha=0.20, linewidth=0)
        phi_cen = np.where(np.isfinite(s['phi_cen']['mean']), s['phi_cen']['mean'], 0)
        ax_cum.plot(mstar_centers, cumulative_above(phi_cen), '-', lw=2.2, **st)
    ax_fsat.set_xlim(9.0, 12.3)
    ax_fsat.set_ylim(0.0, 1.0)
    ax_fsat.grid(True, alpha=0.3)
    ax_fsat.set_ylabel(r'$f_\mathrm{sat}$'  + f'  (z = {z_val:.1f})')
    if i == 0:
        ax_fsat.set_title(r'Satellite fraction $\Phi_\mathrm{sat}/\Phi_\mathrm{total}$')
        ax_fsat.legend(loc='upper right', fontsize=9)
    if i == len(SNAPSHOTS) - 1:
        ax_fsat.set_xlabel(r'$\log_{10}\,M_\star\ [M_\odot/h]$')
    ax_cum.set_yscale('log')
    ax_cum.set_xlim(10.5, 12.3)
    ax_cum.set_ylim(1e-6, 1e-2)
    ax_cum.grid(True, alpha=0.3, which='both')
    if i == 0:
        ax_cum.set_title(r'Cumulative central $N(>M_\star)$')
    if i == len(SNAPSHOTS) - 1:
        ax_cum.set_xlabel(r'$\log_{10}\,M_\star\ [M_\odot/h]$')
    ax_cum.set_ylabel(r'$N(>M_\star)\ [(\mathrm{Mpc}/h)^{-3}]$'  + f'  (z = {z_val:.1f})')
plt.tight_layout()
plt.show()